# Harmony Filtering Service Demo

This notebook shows how to run the core filtering pipeline, using the  
`settings.json` and `config.json` in `config/`.

Before running, make sure:

```bash
pip install -r docs/requirements.txt

In [ ]:
# 1️⃣ Set up imports
import json
from pathlib import Path
import xarray as xr
import matplotlib.pyplot as plt
import resource

from harmony_filtering_service.adapter_utils import load_and_prepare_settings
from harmony_filtering_service.core import process_products
from harmony_filtering_service.identify import identify_dataset

# so that inline plots appear
%matplotlib inline

In [ ]:
# 2️⃣ Load and prepare the settings.json
settings_path = Path("../config/settings.json")
settings = load_and_prepare_settings(settings_path)
print("Data dir:", settings["data_dir"])
print("Output dir:", settings["output_dir"])

In [ ]:
# 3️⃣ Load the filter config (config/config.json)
cfg_path = Path("../config/config.json")
config = json.loads(cfg_path.read_text(encoding="utf-8"))
config

In [ ]:
# 4️⃣ Run the filtering pipeline
#metadata = identify_dataset("MUR25-JPL-L4-GLOB-v04.2", "v4.2")
#process_products(settings, {"MUR": config["MUR"]}, metadata, "20260302090000-JPL-L4_GHRSST-SSTfnd-MUR25-GLOB-v02.0-fv04.2.nc", "analysed_sst")
metadata = identify_dataset("TEMPO_O3TOT_L3", "V04")
process_products(settings, {"O3TOT": config["O3TOT"]}, metadata, "TEMPO_O3TOT_L3_V04_20250912T234443Z_S015.nc", "product/column_amount_o3")
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"Peak memory: {peak / (1024**2):.2f} MB", flush=True)  # B to MB on MacOS
print("✅ Done filtering.")

In [ ]:
from pathlib import Path

test = "MUR25"
if test == "TEMPO":
    file_pattern = "TEMPO*"
    group = "product"
    #var_name = "vertical_column_stratosphere"
    var_name = "column_amount_o3"
elif test == "MUR":
    file_pattern = "*-JPL-L4*fv04.1*"
    group = "/"
    var_name = "analysed_sst"
elif test == "MUR25":
    file_pattern = "*-JPL-L4*fv04.2*"
    group = "/"
    var_name = "analysed_sst"
else:
    file_pattern = "*"
    group = "/"
    var_name = "test"

# 1️⃣ find files
in_dir = Path(settings["data_dir"])
out_dir = Path(settings["output_dir"])
filtered = next(out_dir.glob(f"{file_pattern}_filtered.nc"))
original = in_dir / filtered.name.replace("_filtered.nc", ".nc")

print("Original:", original)
print("Filtered:", filtered)

# 2️⃣ open them
ds_orig = xr.open_dataset(original, group=group)
ds_filt = xr.open_dataset(filtered, group=group)

print(ds_orig)
da_orig = ds_orig[var_name]
da_filt = ds_filt[var_name]

# 3️⃣ helper to get a pure 2D slice
def to_2d(da):
    # if there's a time dimension, grab the first timestep
    if "time" in da.dims:
        da2 = da.isel(time=0)
    else:
        da2 = da.squeeze()
    if da2.ndim != 2:
        raise ValueError(f"Cannot make 2D from dims {da2.dims}")
    return da2


da_o2 = to_2d(da_orig)
da_f2 = to_2d(da_filt)

# 4️⃣ plot side by side with common color scale
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

# first get color limits from the original
vmin, vmax = float(da_o2.min()), float(da_o2.max())

im1 = da_o2.plot.pcolormesh(ax=ax1, add_colorbar=False, vmin=vmin, vmax=vmax)
ax1.set_title("Original")
ax1.set_xlabel("")
ax1.set_ylabel("")

im2 = da_f2.plot.pcolormesh(ax=ax2, add_colorbar=False, vmin=vmin, vmax=vmax)
ax2.set_title("Filtered")
ax2.set_xlabel("")
ax2.set_ylabel("")

# 5️⃣ shared colorbar
cbar = fig.colorbar(im2, ax=(ax1, ax2), orientation="horizontal", pad=0.1)
cbar.set_label(var_name)

plt.show()